In [1]:
import torch 
import torch.nn as nn

import torchvision
from torchvision.transforms import v2
import numpy as np
from torch.utils.data import DataLoader, Dataset, random_split

In [2]:
# dataset_caltech101_v = torchvision.datasets.Caltech101(root='caltech101', download=False)

In [3]:
# accessing data from folder saved in local device
dataset_caltech101 = torchvision.datasets.ImageFolder(root=r'caltech101\data_classes')

In [4]:

# naive split
"""train = int(0.8 * len(dataset_caltech101))
test = len(dataset_caltech101) - train

train_set, test_set = random_split(
    dataset_caltech101, [train, test]
)

print(len(train_set))
print(len(test_set))"""

'train = int(0.8 * len(dataset_caltech101))\ntest = len(dataset_caltech101) - train\n\ntrain_set, test_set = random_split(\n    dataset_caltech101, [train, test]\n)\n\nprint(len(train_set))\nprint(len(test_set))'

In [5]:
len(dataset_caltech101)

8677

In [6]:
# Stratified split: take TRAIN_FRAC of each class (at least 1 sample in each split).
rng = np.random.default_rng(42)
train_idx, test_idx = [], []

labels = np.array(dataset_caltech101.targets)

for c in range(101):
    idx = np.where(labels == c)[0]
    rng.shuffle(idx)
    n_train = min(max(1, int(round(0.8 * len(idx)))), len(idx) - 1)
    train_idx.extend(idx[:n_train].tolist())
    test_idx.extend(idx[n_train:].tolist())

print('train:', len(train_idx), '| test:', len(test_idx))
assert set(train_idx).isdisjoint(test_idx)
assert len(train_idx) + len(test_idx) == len(dataset_caltech101)

train: 6941 | test: 1736


In [7]:
class TransformedSubset(Dataset):
    """A subset of `Base_dataset` that applies its own transform (so train and test can differ)."""
    def __init__(self, Base_dataset, indices, transform):
        self.Base_dataset, self.indices, self.transform = Base_dataset, indices, transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, label = self.Base_dataset[self.indices[i]]
        return self.transform(img), label

In [8]:
IMG_SIZE = 224*224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]    # pretrained imagenet weights
base_data = dataset_caltech101

In [9]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(device)

cuda


In [10]:
train_tfm = v2.Compose([
    v2.RGB(),                                            # handles grayscale and RGBA
    v2.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), antialias=True),
    v2.RandomHorizontalFlip(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])

test_tfm = v2.Compose([                                   # deterministic
    v2.RGB(),
    v2.Resize(256, antialias=True),
    v2.CenterCrop(IMG_SIZE),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])

train_dataset = TransformedSubset(base_data, train_idx, train_tfm)
test_dataset = TransformedSubset(base_data, test_idx, test_tfm)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                          num_workers=0, pin_memory=(device == 'cuda'),
                          drop_last=True)   # avoids a size-1 final batch for BatchNorm
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False,
                         num_workers=0, pin_memory=(device == 'cuda'))

In [11]:
import matplotlib.pyplot as plt

In [12]:
# Visualize a few training samples with class names
mean_t = torch.tensor(MEAN).view(-1, 1, 1)
std_t = torch.tensor(STD).view(-1, 1, 1)

def unnormalize(t):
    return torch.clamp(t * std_t + mean_t, 0, 1)

images, ys = next(iter(train_loader))
fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
for ax, img, y in zip(axes, images[:4], ys[:4]):
    ax.imshow(unnormalize(img).permute(1, 2, 0).numpy())
    ax.set_title(base_data.categories[y.item()], fontsize=9)
    ax.axis('off')
plt.show()

RuntimeError: [enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 30211571712 bytes.

### Model

In [ ]:
class Model(nn.Module):
    pass